In [1]:
import json
import pandas as pd

# The notebook runs from notebooks/, so the data is one folder up
df = pd.read_parquet("../data/final/site_week_table_fe.parquet")

with open("../data/final/feature_sets.json") as f:
    feature_sets = json.load(f)

print("Rows, columns:", df.shape)
print("Feature set names:", list(feature_sets.keys()))


Rows, columns: (21718, 46)
Feature set names: ['labels', 'split_column', 'regional_core', 'regional_candidates', 'site_core', 'site_candidates', 'do_not_use']


In [2]:
# y_class is missing when the next week wasn't sampled, so those rows can't be used for training or scoring
df = df[df["y_class"].notna()].copy()
df["y_class"] = df["y_class"].astype(int)

print("Rows with a label:", len(df))


Rows with a label: 16037


In [3]:
# Rows = train / validate / test / holdout, columns = class 0 / 1 / 2
print(pd.crosstab(df["split"], df["y_class"], margins=True))


y_class       0     1     2    All
split                             
holdout     550     0     0    550
test       1378    49    28   1455
train      9783  1229  1367  12379
validate   1485   123    45   1653
All       13196  1401  1440  16037


In [4]:
features = feature_sets["site_core"]

print("Number of features:", len(features))
print(features)

# How many values are missing in each feature (largest first)
print(df[features].isna().sum().sort_values(ascending=False))


Number of features: 12
['site_last_class', 'site_last_log_cells', 'site_weeks_since_obs', 'nbr_class', 'reg_last_class', 'reg_class_max_4wk', 'reg_n_samples', 'weeks_since_bloom', 'pos_weeks_12wk', 'class_run_weeks', 'season_sin', 'season_cos']
weeks_since_bloom       1111
nbr_class                268
site_last_class          111
site_last_log_cells      111
site_weeks_since_obs     111
reg_class_max_4wk         14
reg_last_class             0
reg_n_samples              0
pos_weeks_12wk             0
class_run_weeks            0
season_sin                 0
season_cos                 0
dtype: int64


In [5]:
import sklearn
print(sklearn.__version__)


1.9.1


In [6]:
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score, confusion_matrix

# A reusable "report card" so every model is graded the same way
def evaluate(name, y_true, y_pred, last_class):
    print(f"===== {name} =====")
    print("Macro-F1:          ", round(f1_score(y_true, y_pred, average="macro"), 3))
    print("Balanced accuracy: ", round(balanced_accuracy_score(y_true, y_pred), 3))

    # Recall = out of all the real weeks of each class, how many did we get right?
    recalls = recall_score(y_true, y_pred, average=None, labels=[0, 1, 2])
    print("Recall class 0 / 1 / 2:", recalls.round(3))

    # Onset = the site had no bloom last time, but next week it does (a NEW bloom)
    onset = (last_class == 0) & (y_true > 0)
    caught = (y_pred[onset] > 0).mean()
    print(f"Onset recall:       {caught:.3f}  (out of {onset.sum()} new blooms)")

    # Confusion matrix: rows = what really happened, columns = what we predicted
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    print(pd.DataFrame(cm, index=["true 0", "true 1", "true 2"],
                       columns=["pred 0", "pred 1", "pred 2"]))
    print()


Split Data by Year

In [7]:
# Learn from the older years, check on 2024 (don't touch 2025 yet)
train = df[df["split"] == "train"]
val = df[df["split"] == "validate"]

X_train, y_train = train[features], train["y_class"]   # X = features (clues), y = answer
X_val, y_val = val[features], val["y_class"]

print("Train rows:", len(train), "| Validate rows:", len(val))


Train rows: 12379 | Validate rows: 1653


 Baseline, "next week = last week"

In [8]:
# No learning: just copy the site's last class. If there's no last class, guess 0 (no bloom)
baseline_pred = val["site_last_class"].fillna(0).astype(int)

evaluate("Baseline: next week = last week", y_val, baseline_pred, val["site_last_class"])


===== Baseline: next week = last week =====
Macro-F1:           0.49
Balanced accuracy:  0.478
Recall class 0 / 1 / 2: [0.957 0.301 0.178]
Onset recall:       0.000  (out of 90 new blooms)
        pred 0  pred 1  pred 2
true 0    1421      51      13
true 1      71      37      15
true 2      19      18       8



Train random Forest

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,          # 500 trees that vote
    class_weight="balanced",   # count the rare bloom weeks more heavily
    min_samples_leaf=10,       # each tree "rule" must cover at least 10 rows (prevents memorizing)
    random_state=42,           # same result every time you run it
    n_jobs=-1,                 # use all your computer's cores to go faster
)

rf.fit(X_train, y_train)       # learn from the training years
rf_pred = rf.predict(X_val)    # predict 2024

evaluate("Random forest (12 core features)", y_val, rf_pred, val["site_last_class"])


===== Random forest (12 core features) =====
Macro-F1:           0.472
Balanced accuracy:  0.499
Recall class 0 / 1 / 2: [0.91  0.341 0.244]
Onset recall:       0.511  (out of 90 new blooms)
        pred 0  pred 1  pred 2
true 0    1352      45      88
true 1      30      42      51
true 2      14      20      11

